# Reconstruction lab

Reconstruct from the **registered operators** (`results/recon/ops/<name>/`), which `build_operator.py` writes when a build finishes. Nothing here rebuilds A; each reconstruction is a PnP solve on the cached matrix (the first use of an operator loads ~5.4 GB into RAM).

- **Equal dose** (default): every coder's illumination normalised to the same mean.
- **Equal laser power** (`power=True`): A scaled by the light each coder keeps through the NA-0.8 relay, noise fixed at the reference coder's level.
- Measurements are taken on the fine 0.32 um grid at build time, so there is no inverse crime; `measure='grid'` gives the A @ GT upper bound.

Engine: `notebooks/reconlab.py`. Priors: `preset='dncnn' | 'l1' | 'no_z' | 'xy_l1'` plus any `reconstruct.RECON` knob as a keyword.

In [ ]:
%matplotlib inline
import os, sys
p = os.getcwd()
while p != os.path.dirname(p) and not os.path.isdir(os.path.join(p, 'pipeline')):
    p = os.path.dirname(p)
os.chdir(p); sys.path[:0] = [p, os.path.join(p, 'notebooks')]
import reconlab
lab = reconlab.Lab()

## 1. One reconstruction

In [ ]:
OPS = list(lab.meta)
est, m = lab.recon(OPS[0], specimen='neural_cell', snr=25, preset='dncnn')

## 2. Compare coders: same specimen, same noise draw, same prior
Includes the depth-blind oracle (exact lateral structure, exact global axial profile, no local depth): a coder beats it only if it recovers depth locally.

In [ ]:
ests, rows = lab.compare(OPS, specimen='neural_cell', snr=20, preset='dncnn')

In [ ]:
ests, rows = lab.compare(OPS, specimen='neural_cell', snr=20, preset='dncnn', power=True)   # equal laser power

## 3. The mirror ghost: bead phantom
Beads in mirror pairs about the focal plane. ghost = energy at the mirror depth / at the true depth (0 clean, ~1 degenerate).

In [ ]:
rows = lab.beads(OPS, snr=25, preset='l1')

## 4. Interactive panel
Pick operators and settings, then press **Run** (each solve takes a while, so it runs on demand).

In [ ]:
from ipywidgets import interact_manual, SelectMultiple, Dropdown, FloatSlider, IntSlider, Checkbox
interact_manual(lambda ops, specimen, snr, preset, power, seed, max_iter:
                    lab.compare(list(ops), specimen=specimen, snr=snr, preset=preset, power=power,
                                seed=seed, max_iter=max_iter),
                ops=SelectMultiple(options=OPS, value=tuple(OPS[:2])),
                specimen=Dropdown(options=list(lab.gt), value='neural_cell'),
                snr=FloatSlider(value=20, min=5, max=40, step=2.5),
                preset=Dropdown(options=list(reconlab.PRESETS), value='dncnn'),
                power=Checkbox(value=False, description='equal laser power'),
                seed=IntSlider(0, 0, 9), max_iter=IntSlider(20, 5, 100, step=5));

## 5. Predicted vs actual (keep checking the metric)
The Wiener prediction is validated for **ranking** only; it misses the low-SNR crossover. Compare with real reconstructions over SNR.

In [ ]:
pred = lab.predict(OPS, snrs=(10, 15, 20, 25, 30))
for snr in (10, 20, 30):
    for op in OPS:
        lab.solve(op, 'neural_cell', snr=snr, preset='dncnn')

## 6. Views

In [ ]:
lab.slices(OPS[0], specimen='neural_cell', snr=25, preset='l1')    # per-depth montage vs GT
lab.error(OPS[0], specimen='neural_cell', snr=25, preset='l1')     # |est-GT| + voxel scatter
lab.converge(OPS[0], specimen='neural_cell', snr=25, preset='l1')  # ADMM residuals

In [ ]:
lab.measurements(OPS[0], 'neural_cell', snr=20)   # the detector frames (first 64 patterns)
lab.depth_corr(OPS[0])                            # A's depth-column correlation

## 7. Play: any prior knob, fewer patterns
`patterns=k` keeps the first k patterns of the scan (fewer camera frames).

In [ ]:
for l1 in (0.02, 0.05, 0.1, 0.2):
    lab.solve(OPS[0], 'beads', snr=25, preset='l1', l1_lmda=l1)
for k in (64, 128, 256):
    lab.solve(OPS[0], 'neural_cell', snr=25, preset='dncnn', patterns=k)